# Step 5b — DL: GNN on native-space point cloud

**Motivation:** the 3D CNN (notebook 05) resamples the sparse point cloud onto a fixed grid using nearest-neighbour interpolation. Because the source has ~470 voxels but the target grid has ~1176 active cells, most grid cells are assigned from a non-collocated source point — systematic artefacts that the CNN then has to unlearn.

**This notebook** operates directly on the raw `(w, u, v, I)` point cloud with no resampling:
- Each voxel is a graph **node** with features `(w_norm, u, v, r, I)` where `w_norm ∈ [0,1]` (relative axial position) and `r = sqrt(u²+v²)` (radial distance).
- **Edges** connect each node to its k=8 nearest neighbours in `(w_norm, u, v)` space — the same physical neighbourhood used for the k-NN gradient features in the baseline.
- Three **EdgeConv** layers learn local aggregation in the actual needle-coordinate geometry.
- **Global mean + max pooling** (concatenated) produces a permutation-invariant graph embedding.
- Same train/test split, same 5-fold CV, same label-smoothed weighted BCE as the 3D CNN.

**Augmentations (applied to raw coords before graph construction):**
- Random rotation around w-axis (continuous, exact in needle space)
- Random radial flip
- Intensity jitter

In [ ]:
# ── Colab setup (skipped if packages already present) ──────────────────────────
try:
    import torch_geometric
except ImportError:
    import subprocess, torch
    v   = torch.__version__.split('+')[0]
    cu  = ('cu' + torch.version.cuda.replace('.', '')) if torch.cuda.is_available() else 'cpu'
    url = f"https://data.pyg.org/whl/torch-{v}+{cu}.html"
    subprocess.run(['pip', 'install', '-q', 'torch_geometric',
                    'torch_scatter', 'torch_sparse', 'torch_cluster', '-f', url],
                   check=True)
    print('PyG installed — restart runtime if prompted.')


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torch_geometric.nn as pyg_nn
from torch_geometric.data import Data, Batch
from torch_cluster import knn_graph

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.notebook import tqdm

REPO_ROOT   = Path('..').resolve()
PATCHES_DIR = REPO_ROOT / 'data' / 'patches'
EDGES_DIR   = REPO_ROOT / 'data' / 'edges'
MANIFEST    = REPO_ROOT / 'data' / 'manifest.csv'
DROP_LIST   = REPO_ROOT / 'data' / 'cores_to_drop_contamination.csv'
TEST_SUBJS  = REPO_ROOT / 'data' / 'test_subjects.csv'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# TF32 on A100 gives ~3× matmul speedup with negligible precision loss
if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')

In [ ]:
# ── Colab data setup ────────────────────────────────────────────────────────────
# Copies patches + CSVs from Google Drive to the local Colab SSD (/content/data)
# so that edge precomputation and dataset init run on fast local I/O.
#
# On Drive, put the data folder at:  MyDrive/gleason_data/
#   gleason_data/
#     manifest.csv
#     cores_to_drop_contamination.csv
#     test_subjects.csv
#     patches/   (*.npz files)
#
# Skip this cell if running locally — path variables from the cell above are used.

import os, shutil, subprocess

if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    DRIVE_DATA = Path('/content/drive/MyDrive/gleason_data')
    LOCAL_DATA = Path('/content/data')
    LOCAL_DATA.mkdir(exist_ok=True)

    # Copy patches to local SSD (16k files, ~125 MB — takes ~30s from Drive)
    local_patches = LOCAL_DATA / 'patches'
    if not local_patches.exists():
        print('Copying patches to local SSD...', end=' ', flush=True)
        subprocess.run(['cp', '-r', str(DRIVE_DATA / 'patches'), str(local_patches)], check=True)
        print(f'done ({len(list(local_patches.glob("*.npz")))} files).')
    else:
        print(f'Patches already on local SSD ({len(list(local_patches.glob("*.npz")))} files).')

    # Copy CSVs
    for fname in ['manifest.csv', 'cores_to_drop_contamination.csv', 'test_subjects.csv']:
        dst = LOCAL_DATA / fname
        if not dst.exists():
            shutil.copy(DRIVE_DATA / fname, dst)

    # Override path variables — everything below uses local SSD
    PATCHES_DIR = LOCAL_DATA / 'patches'
    EDGES_DIR   = LOCAL_DATA / 'edges'
    MANIFEST    = LOCAL_DATA / 'manifest.csv'
    DROP_LIST   = LOCAL_DATA / 'cores_to_drop_contamination.csv'
    TEST_SUBJS  = LOCAL_DATA / 'test_subjects.csv'

    print(f'Data paths → {LOCAL_DATA}')


## 1. Load manifest & splits

In [25]:
manifest = pd.read_csv(MANIFEST)
drop_ids = set(pd.read_csv(DROP_LIST)['core_id'])
manifest = manifest[~manifest['core_id'].isin(drop_ids)].reset_index(drop=True)

extracted = {int(p.stem) for p in PATCHES_DIR.glob('*.npz')}
manifest  = manifest[manifest['core_id'].isin(extracted)].reset_index(drop=True)

test_subjs = set(pd.read_csv(TEST_SUBJS, header=None)[0].tolist())
manifest['split'] = manifest['subject_id'].map(
    lambda s: 'test' if s in test_subjs else 'trainval'
)

for split in ['trainval', 'test']:
    sub = manifest[manifest['split'] == split]
    pos = (sub['label'] == 1).sum()
    print(f'{split:9s}: {len(sub):6,d} cores  ({sub.subject_id.nunique():3d} subjects)'
          f'  pos={pos} ({100*pos/len(sub):.1f}%)')

trainval : 14,192 cores  (676 subjects)  pos=521 (3.7%)
test     :  2,498 cores  (120 subjects)  pos=126 (5.0%)


## 2. Point-cloud → PyG graph

Node features: `(w_norm, u, v, r, I)` — 5 dimensions.  
Edges: k=6 nearest neighbours in `(w_norm, u, v)` space, precomputed to `data/edges/`.  
All `Data` objects are built at dataset init; `__getitem__` only clones `x` and applies tensor-only augmentation.

In [ ]:
K_NEIGHBORS = 8

In [27]:
## 2b. Precompute edge indices (run once; re-run if K_NEIGHBORS changes)
#
# knn_graph is called ~14k times per epoch if done at training time.
# Rotation around w-axis and radial flip are both isometric in (w_norm, u, v)
# → pairwise distances unchanged → edge index is identical after augmentation.
# Saved to data/edges/{core_id}.npy  (shape: 2 × N*K_NEIGHBORS*2)

import shutil
EDGES_DIR.mkdir(parents=True, exist_ok=True)

# Detect stale edges from a different K by checking one file's edge count
_sample_cid = manifest['core_id'].iloc[0]
_sample_edge_path = EDGES_DIR / f'{_sample_cid}.npy'
_stale = False
if _sample_edge_path.exists():
    _ei = np.load(_sample_edge_path)
    _d  = np.load(PATCHES_DIR / f'{_sample_cid}.npz')
    _expected_edges = _d['coords'].shape[0] * K_NEIGHBORS * 2
    if _ei.shape[1] != _expected_edges:
        print(f'K_NEIGHBORS changed — clearing stale edges ({_ei.shape[1]} → {_expected_edges})')
        shutil.rmtree(EDGES_DIR)
        EDGES_DIR.mkdir()
        _stale = True

missing = [cid for cid in manifest['core_id']
           if not (EDGES_DIR / f'{cid}.npy').exists()]
print(f'Cores to precompute: {len(missing)} / {len(manifest)}')

for cid in tqdm(missing, desc='Precomputing edges'):
    d     = np.load(PATCHES_DIR / f'{cid}.npz')
    w     = d['coords'][:, 0]
    u     = d['coords'][:, 1]
    v     = d['coords'][:, 2]
    w_norm = (w - w.min()) / (w.max() - w.min() + 1e-8)
    pos   = torch.tensor(np.stack([w_norm, u, v], axis=1), dtype=torch.float32)
    ei    = knn_graph(pos, k=K_NEIGHBORS, loop=False).numpy()
    np.save(EDGES_DIR / f'{cid}.npy', ei)

print('Done.')


K_NEIGHBORS changed — clearing stale edges (2886 → 5772)
Cores to precompute: 16690 / 16690


Precomputing edges:   0%|          | 0/16690 [00:00<?, ?it/s]

Done.


## 3. Dataset

In [28]:
import math

class BiopsyGraphDataset(Dataset):
    """
    Precomputes all Data objects at init (one-time cost).
    __getitem__ only clones x and applies tensor-only augmentation — no numpy, no disk I/O.
    """

    def __init__(self, core_ids, labels, patches_dir, edges_dir, augment=False):
        self.augment = augment
        patches_dir  = Path(patches_dir)
        edges_dir    = Path(edges_dir)

        self.graphs = []
        for cid, label in zip(core_ids, labels):
            d      = np.load(patches_dir / f'{cid}.npz')
            ei     = np.load(edges_dir   / f'{cid}.npy')
            coords = d['coords']
            intens = d['intensity']

            w      = coords[:, 0]
            u      = coords[:, 1]
            v      = coords[:, 2]
            w_norm = (w - w.min()) / (w.max() - w.min() + 1e-8)
            r      = np.sqrt(u**2 + v**2)

            self.graphs.append(Data(
                x          = torch.tensor(
                                 np.stack([w_norm, u, v, r, intens], axis=1),
                                 dtype=torch.float32),
                edge_index = torch.tensor(ei, dtype=torch.long),
                y          = torch.tensor([float(label)], dtype=torch.float32),
            ))

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        g = self.graphs[idx]
        if not self.augment:
            return g

        # Tensor-only augmentation — x layout: [w_norm, u, v, r, I]
        x = g.x.clone()   # (N, 5)

        # Rotation around w-axis (preserves w_norm and r)
        theta    = torch.empty(1).uniform_(0, 2 * math.pi).item()
        c, s     = math.cos(theta), math.sin(theta)
        u, v     = x[:, 1].clone(), x[:, 2].clone()
        x[:, 1]  =  c * u - s * v
        x[:, 2]  =  s * u + c * v

        # Radial flip (preserves r)
        if torch.rand(1).item() < 0.5:
            x[:, 1] = -x[:, 1]

        # Intensity jitter
        x[:, 4] = x[:, 4] + torch.randn(1).item() * 0.05
        x[:, 4] = x[:, 4] * (1.0 + torch.randn(1).item() * 0.05)

        return Data(x=x, edge_index=g.edge_index, y=g.y)


def collate_graphs(batch):
    return Batch.from_data_list(batch)

## 4. Model — EdgeConv GNN (DGCNN)

**EdgeConv** (Wang et al. 2019): message is `MLP(cat(x_i, x_j − x_i))` — captures both absolute features and local contrast between neighbours. The `x_j − x_i` term is the graph analogue of the gradient features in the baseline, and is the primary motivation for using a GNN on texture data.

`aggr='max'` uses `scatter_max` which has native CUDA kernels on GPU.

Verified shape trace:
```
Input node features              (N_total, 5)
EdgeConv1  mlp(10→32),  aggr=max (N_total, 32)
EdgeConv2  mlp(64→64),  aggr=max (N_total, 64)
EdgeConv3  mlp(128→64), aggr=max (N_total, 64)
global_mean_pool + global_max_pool, cat  (B, 128)
Linear(128→32)+ReLU+Dropout(0.4)
Linear(32→1) → squeeze           (B,)
```

In [ ]:
from torch_geometric.nn import global_mean_pool, global_max_pool


def mlp(in_ch, out_ch):
    """Two-layer MLP used inside each EdgeConv."""
    return nn.Sequential(
        nn.Linear(in_ch,  out_ch), nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
        nn.Linear(out_ch, out_ch), nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
    )


class BiopsyGNN(nn.Module):
    def __init__(self, in_channels: int = 5):
        super().__init__()
        self.conv1 = pyg_nn.EdgeConv(mlp(2 * in_channels, 32), aggr='max')
        self.conv2 = pyg_nn.EdgeConv(mlp(2 * 32,          64), aggr='max')
        self.conv3 = pyg_nn.EdgeConv(mlp(2 * 64,          64), aggr='max')

        self.head = nn.Sequential(
            nn.Linear(128, 32), nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, 1),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)   # (N_total, 32)
        x = self.conv2(x, edge_index)   # (N_total, 64)
        x = self.conv3(x, edge_index)   # (N_total, 64)

        x = torch.cat([global_mean_pool(x, batch),
                        global_max_pool(x, batch)], dim=1)  # (B, 128)

        return self.head(x).squeeze(1)  # (B,)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


_m = BiopsyGNN()
print(f'Parameters: {count_params(_m):,}')

# Smoke-test
_row = manifest.iloc[0]
_ds  = BiopsyGraphDataset([_row.core_id, _row.core_id],
                           [float(_row.label), float(_row.label)],
                           PATCHES_DIR, EDGES_DIR, augment=False)
_batch = collate_graphs([_ds[0], _ds[1]])
with torch.no_grad():
    _out = _m(_batch)
print(f'Output shape: {tuple(_out.shape)}  ✓')

## 5. Training utilities

In [ ]:
LOG_EVERY    = 10
LABEL_SMOOTH = 0.05   # GG2/GG3 boundary has known inter-observer variability


def run_epoch(model, loader, optimizer, pos_weight, device, train=True):
    model.train(train)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    total_loss, all_probs, all_labels = 0.0, [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            y      = batch.y.squeeze()

            y_smooth = y * (1 - 2 * LABEL_SMOOTH) + LABEL_SMOOTH if train else y
            loss = criterion(logits, y_smooth)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(y)
            all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.append(y.detach().cpu().numpy())

    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    auc    = roc_auc_score(labels, probs) if labels.sum() > 0 else 0.5
    return total_loss / len(loader.dataset), auc, probs


def train_fold(
    train_ids, train_labels,
    val_ids,   val_labels,
    patches_dir, edges_dir, device,
    n_epochs=150, batch_size=128, lr=1e-3,
    patience=30, weight_decay=5e-3,
):
    n_pos = int(train_labels.sum())
    n_neg = int((1 - train_labels).sum())
    pw    = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)

    pin = (device.type == 'cuda')
    train_ds = BiopsyGraphDataset(train_ids, train_labels, patches_dir, edges_dir, augment=True)
    val_ds   = BiopsyGraphDataset(val_ids,   val_labels,   patches_dir, edges_dir, augment=False)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          collate_fn=collate_graphs, num_workers=0, pin_memory=pin)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                          collate_fn=collate_graphs, num_workers=0, pin_memory=pin)

    model     = BiopsyGNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_auc, best_probs, no_improve = 0.0, None, 0
    history = []

    for epoch in range(n_epochs):
        tr_loss, tr_auc, _      = run_epoch(model, train_dl, optimizer, pw, device, train=True)
        va_loss, va_auc, v_prob = run_epoch(model, val_dl,   optimizer, pw, device, train=False)
        scheduler.step()
        history.append((tr_loss, tr_auc, va_loss, va_auc))

        if (epoch + 1) % LOG_EVERY == 0:
            print(f'  ep {epoch+1:3d}  '
                  f'tr loss={tr_loss:.4f} auc={tr_auc:.3f}  |  '
                  f'val loss={va_loss:.4f} auc={va_auc:.3f}')

        if va_auc > best_auc:
            best_auc, best_probs, no_improve = va_auc, v_prob, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  early stop at epoch {epoch+1}')
                break

    return best_auc, best_probs, val_labels, history, best_state

## 6. 5-fold cross-validation

In [ ]:
tv_df = manifest[manifest['split'] == 'trainval'].reset_index(drop=True)

core_ids = tv_df['core_id'].values
labels   = tv_df['label'].values
groups   = tv_df['subject_id'].values

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

fold_aucs, fold_aps, all_histories = [], [], []

for fold, (tr_idx, va_idx) in enumerate(cv.split(core_ids, labels, groups=groups)):
    print(f'\nFold {fold+1}/5  '
          f'train pos={labels[tr_idx].sum()}  val pos={labels[va_idx].sum()}')

    best_auc, val_probs, val_labels, history, _ = train_fold(
        core_ids[tr_idx], labels[tr_idx].astype(np.float32),
        core_ids[va_idx], labels[va_idx].astype(np.float32),
        PATCHES_DIR, EDGES_DIR, DEVICE,
        n_epochs=150, batch_size=128, lr=1e-3, patience=30,
    )
    ap = average_precision_score(val_labels, val_probs)
    fold_aucs.append(best_auc)
    fold_aps.append(ap)
    all_histories.append(history)
    print(f'  → best val AUC={best_auc:.3f}  AP={ap:.3f}')

print(f'\nCV ROC-AUC : {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}')
print(f'CV PR-AUC  : {np.mean(fold_aps):.3f} ± {np.std(fold_aps):.3f}')

### Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = plt.cm.tab10(np.linspace(0, 0.5, 5))

for fold, hist in enumerate(all_histories):
    hist = np.array(hist)
    ep   = np.arange(1, len(hist) + 1)
    axes[0].plot(ep, hist[:, 0], color=colors[fold], alpha=0.7)
    axes[0].plot(ep, hist[:, 2], color=colors[fold], alpha=0.7, linestyle='--')
    axes[1].plot(ep, hist[:, 1], color=colors[fold], alpha=0.7, label=f'fold {fold+1} train')
    axes[1].plot(ep, hist[:, 3], color=colors[fold], alpha=0.7, linestyle='--')

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE loss')
axes[0].set_title('Training (—) vs validation (--) loss')
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Training (—) vs validation (--) AUC')
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Final model — retrain on all trainval, evaluate on test

In [ ]:
test_df  = manifest[manifest['split'] == 'test'].reset_index(drop=True)
test_ids = test_df['core_id'].values
y_test   = test_df['label'].values

median_epochs = int(np.median([len(h) for h in all_histories]))
final_epochs  = max(50, int(median_epochs * 1.1))
print(f'CV median stopping epoch: {median_epochs}  → final training: {final_epochs} epochs')

n_pos = int(labels.sum())
n_neg = int((1 - labels).sum())
pw    = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
pin   = (DEVICE.type == 'cuda')

train_ds = BiopsyGraphDataset(core_ids, labels.astype(np.float32), PATCHES_DIR, EDGES_DIR, augment=True)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True,
                      collate_fn=collate_graphs, num_workers=0, pin_memory=pin)

final_model = BiopsyGNN().to(DEVICE)
optimizer   = torch.optim.Adam(final_model.parameters(), lr=1e-3, weight_decay=5e-3)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=final_epochs)

for epoch in range(final_epochs):
    tr_loss, tr_auc, _ = run_epoch(final_model, train_dl, optimizer, pw, DEVICE, train=True)
    scheduler.step()
    if (epoch + 1) % LOG_EVERY == 0:
        print(f'  ep {epoch+1:3d}  loss={tr_loss:.4f}  auc={tr_auc:.3f}')

test_ds = BiopsyGraphDataset(test_ids, y_test.astype(np.float32), PATCHES_DIR, EDGES_DIR, augment=False)
test_dl = DataLoader(test_ds, batch_size=256, shuffle=False,
                     collate_fn=collate_graphs, num_workers=0, pin_memory=pin)
_, test_auc, test_probs = run_epoch(final_model, test_dl, None, pw, DEVICE, train=False)
test_ap = average_precision_score(y_test, test_probs)

print(f'\nTest ROC-AUC : {test_auc:.3f}')
print(f'Test PR-AUC  : {test_ap:.3f}')

## 8. Comparison

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

print(f'{"Model":<30} {"CV ROC-AUC":>12} {"Test ROC-AUC":>13} {"Test PR-AUC":>12}')
print('-' * 70)
print(f'{"Logistic Regression (BL)":<30} {"0.596±0.044":>12} {"0.583":>13} {"0.070":>12}')
print(f'{"Gradient Boosting (BL)":<30} {"0.571±0.036":>12} {"0.571":>13} {"0.070":>12}')
print(f'{"3D CNN (nb 05)":<30} {"(see nb 05)":>12} {"(see nb 05)":>13} {"":>12}')
print(f'{"GNN EdgeConv (this nb)":<30} '
      f'{f"{np.mean(fold_aucs):.3f}±{np.std(fold_aucs):.3f}":>12}'
      f' {test_auc:>13.3f} {test_ap:>12.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot([0,1],[0,1],'--', color='gray', linewidth=0.8)
fpr, tpr, _ = roc_curve(y_test, test_probs)
ax.plot(fpr, tpr, color='#9C27B0', linewidth=2,
        label=f'GNN EdgeConv  (AUC={test_auc:.3f})')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC — Test set'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
baseline_pr = y_test.mean()
ax.axhline(baseline_pr, linestyle='--', color='gray', linewidth=0.8,
           label=f'Random  (P={baseline_pr:.3f})')
prec, rec, _ = precision_recall_curve(y_test, test_probs)
ax.plot(rec, prec, color='#9C27B0', linewidth=2,
        label=f'GNN EdgeConv  (AP={test_ap:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall — Test set'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/gnn_curves.png', dpi=150, bbox_inches='tight')
plt.show()